# 2\. Ingesting and Cleaning All Datasets

Purpose: This notebook ingests all raw datasets from their respective sources, conduct basic cleaning like converting data types and dropping some rows with NaN values, and writing the cleaned dfs to csv for easy downstream data manipulation\. 

Inputs
1\. health\_rankings\_2025\.xlsx : uploaded Excel file of 2025 county health rankings\. While it is possible to read directly from the file's download url; however, due to inconsistent file naming conventions on the County Health Rankings webpage, it is more reliable to read the data directly from the file itself\. 
2\. chd\_metrics\_config\_OLS1\.csv : a config file that enables the user to easily rename columns\. 

Outputs
1\. county\_to\_county\_US\_2020\.csv : csv file of 2020 US Census Bureau county\-to\-county migration data
2\. cleaned\_health\_rankings\_2025\.csv : csv file of county health rankings and associated z\-scores
3\. cleaned\_chd\_metrics\_2022\.csv : csv file of the individual metrics that contribute to county health rankings 

In [1]:
import os
import io
from pathlib import Path

import requests
import pandas as pd
import re

## 1\. US Census Migration Dataset

Short description: US Census Bureau’s American Community Survey \(ACS\) 5\-Year Migration Flow\. It measures migration primarily by residents’ survey responses regarding where they lived when surveyed and where they lived one year ago\. The data is collected over a 5\-year period to ensure larger samples for smaller\-population counties and then used to produce an annual average\. The data below is from the 2020 dataset, meaning that it incorporates survey responses collected between 2016 and 2020\. We selected this dataset because it is the last one with county\-to\-county migration flows\. 

Location: The US Census Bureau's Migration Flow Datasets can be found here: https://www\.census\.gov/data/developers/data\-sets/acs\-migration\-flows\.html 

Method: The data is ingested via an API provided by the US Census Bureau\. 

Key Features \(full list of variables can be found here: https://api\.census\.gov/data/2020/acs/flows/variables\.html\)
GEOID1: First County's FIPS Code \(FIPS Codes are the key by which we will merge datasets\)\. 
GEOID2: Second County's FIPS Code
FULL1\_NAME: First County's Name and State
FULL2\_NAME: Second County's Name and State
STATE1\_NAME: First County's State
STATE2\_NAME: Second County's State
MOVEDIN: Average number of people who moved from the second county to the first county in one year\. 
MOVEDOUT: Average number of people who moved from the first county to the second county in one year\. 
MOVEDNET: MOVEDIN \- MOVEDOUT
POP1YR: Population of the first county in 2020\. 
POP1YRAGO: Population of the first county in 2019\. 

### Ingesting the Migration Data

*Note - To read in via API you will need to get your own API Key for the US Census from https://api.census.gov/data/key_signup.html*. Alternatively, the raw data is stored as raw_migration_data.csv.

In [2]:
def read_in_migration_data():
    
    ## Identify the API's url and the api_key (as required by the Census Bureau)
    base_url = "https://api.census.gov/data/2020/acs/flows"
    api_key = os.environ.get("CENSUS_API_KEY")

    ## Identify the desired variables and scope. 
    params = {
        "get": "GEOID1,GEOID2,FULL1_NAME,FULL2_NAME,STATE1_NAME,STATE2_NAME,MOVEDIN,MOVEDOUT,MOVEDNET,POP1YR,POP1YRAGO",
        "for": "county:*", # * ingests all counties
        "in": "state:*", # * ingests all states. 56 is just Wyoming. 
        "key": api_key
    }

    ## Call the API
    response = requests.get(base_url, params=params)

    ## If the API call worked as expected and extract the data in JSON format. 
    if response.status_code == 200: # make sure the response is "OK"
        data = response.json()  # grab the JSON
    else:
        raise Exception('API call unsuccessful')
        
    ## Create a DataFrame
    headers = data[0]
    migration_df = pd.DataFrame(data[1:], columns=headers)

    return migration_df

In [3]:
raw_mig_file_path = 'data/raw_migration_data.csv'

try:
    migration_df = pd.read_csv(raw_mig_file_path, dtype = {'GEOID1': str, 'GEOID2': str, 'state': str, 'county': str})

except:
    print('No cached datasest found. Attempting Census API call.')
    try:

        migration_df = read_in_migration_data()
        migration_df.to_csv(raw_mig_file_path, index=False)

        print('Census API call successfull. Writing raw data to csv')

    except:
        
        raise Exception ('No Census API key found. Please request a Census API key via https://api.census.gov/data/key_signup.html.')


### Cleaning the Migration Dataset

In [4]:
## Select method for error_handling by uncommenting one of the following lines.
error_handling = 'coerce' 
# error_handling = 'raise'
# error_handling = 'ignore'

def clean_migration_df(df=migration_df):

    ## Convert numerical columns to type float for downstream analysis. 
    numeric_columns = ['MOVEDIN','MOVEDOUT','MOVEDNET','POP1YR','POP1YRAGO']
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors=error_handling)
        df[col] = df[col].astype(float)

    ## Filter to only the county-to-county flows.
    # States and countries have FIPS codes less than 5 characters. 
    df = df[df['GEOID1'].str.len() == 5] # GEOID1 is a county
    df = df[df['GEOID2'].str.len() == 5] # GEOID2 is a county

    return df

In [5]:
cleaned_migration_df = clean_migration_df(migration_df)

Note: The US Census Bureau automatically interprets no reported migration between counties as zero migration between counties\. Given the number of sparsely populated counties in the US, this is a widely accepted method for handling 'missing' data\. The assert statements simply confirm there are no missing values in the migration dataset for us to handle\. Lucky for us\!  

In [6]:
def migration_df_asserts(df=cleaned_migration_df):

    # Check for NaN values of any kind
    assert df.isna().any().any() == False

    # Before checking for duplicates, ensure that all counties FIPS codes are the correct five-character length. 
    assert (df['GEOID1'].str.len() == 5).all()
    assert (df['GEOID2'].str.len() == 5).all()

    # Check for duplicate rows
    assert df.duplicated(subset=['GEOID1', 'GEOID2']).any() == False
    # Verify that county-to-county pairs with zero in- or out-migration are excluded
    zero_both = (df['MOVEDIN'] == 0) & (df['MOVEDOUT'] == 0)
    assert zero_both.sum() == 0

    print('No missing values, duplicate rows, or all-zero counties found!')
    print('cleaned_migration_df is ready to write to csv.')
    
    return None

In [7]:
## Run asserts and then write to csv for downstream data manipulation. 

migration_df_asserts(cleaned_migration_df)

cleaned_migration_df.to_csv('data/county_to_county_US_2020.csv', index=False)

No missing values, duplicate rows, or all-zero counties found!
cleaned_migration_df is ready to write to csv.


## 2\. County Health Data: Overall Rankings

Short description: 2025 County Health Release National Data as published by the University of Wisconsin Population Health Institute\. This specific file lists two z\-scores for each county, one for population health and well\-being, and another for community conditions\. The population health and well\-being z\-score is a weighted sum of 5 individual metrics measuring length and quality of life\. The community conditions z\-score is a weighted sum of 24 individual metrics measuring health infrastructure, the physical environment, and social and economic factors\. Greater detail on their methods can be found in their technical document: https://www\.countyhealthrankings\.org/sites/default/files/media/document/CHRR%20Technical%20Documentation%202025\_2\.pdf\. 

Location: County Health Rankings Data and Documentation can be found here: https://www\.countyhealthrankings\.org/health\-data/methodology\-and\-sources/data\-documentation
File Link: From that webpage, the z\-scores xlsx can be downloaded via this link: https://www\.countyhealthrankings\.org/sites/default/files/media/document/2025%20County%20Health%20Rankings%20Data%20\-%20v4\.xlsx

Method: Use file link and read in via pandas read\_excel method\.

Key Features: 
FIPS: County FIPS Code \(used to merge with migration data\)
National Z\-Score: Population Health and Well\-Being Z\-Score
National Z\-Score: Community Conditions Z\-Score
\* Because it is an Excel Document, they are able to duplicate column titles\. We will rename\. 

### Ingesting the County Health Rankings

In [8]:
def read_in_county_health_rankings():

    # Read the Excel file
    health_df = pd.read_excel('data/health_rankings_2025.xlsx', sheet_name='Health Groups', header=1)

    return health_df

In [9]:
health_df = read_in_county_health_rankings()

### Cleaning the County Health Rankings

In [10]:
def clean_health_df(df=health_df):

    # Remove whitespace on column names
    df.columns = df.columns.str.strip()

    # Rename z-score columns 
    df.rename(columns={
        'National Z-Score': 'Health_Z_Score', 
        'National Z-Score.1': 'Community_Z_Score'
    }, inplace=True)

    # Drop rows missing the target data. 
    df = df.dropna(subset=['County'])
    df = df.dropna(subset=['Health_Z_Score','Community_Z_Score'])
        
    # Convert z-scores to numeric 
    df['Health_Z_Score'] = pd.to_numeric(df['Health_Z_Score'], errors=error_handling)
    df['Community_Z_Score'] = pd.to_numeric(df['Community_Z_Score'], errors=error_handling)

    # Multiple z-scores by -1 so that higher z-scores equate to 'better' health outcomes. 
    df['Health_Z_Score'] = df['Health_Z_Score'] * -1
    df['Community_Z_Score'] = df['Community_Z_Score'] * -1

    # Convert to string, remove decimals if any, and pad with zeros to length 5
    df['FIPS'] = df['FIPS'].astype(int).astype(str).str.zfill(5)

    return df

In [11]:
cleaned_health_df = clean_health_df(health_df)

cleaned_health_df.to_csv('data/cleaned_health_rankings_2025.csv')

In [12]:
cleaned_health_df.head()

,FIPS,State,County,Number of Counties Included in Health Groups,Health_Z_Score,Health Group,Health Group Range,Community_Z_Score,Health Group.1,Health Group Range.1
1,01001,Alabama,Autauga,67,-0.040894,5.0,-0.05 to 0.27,0.114428,5.0,-0.17 to 0.03
2,01003,Alabama,Baldwin,67,0.309818,4.0,-0.38 to -0.06,0.386812,3.0,-0.58 to -0.37
3,01005,Alabama,Barbour,67,-1.116797,8.0,0.96 to 1.38,-0.822974,9.0,0.73 to 1.09
4,01007,Alabama,Bibb,67,-0.665055,7.0,0.6 to 0.95,-0.594501,8.0,0.47 to 0.72
5,01009,Alabama,Blount,67,-0.356964,6.0,0.28 to 0.6,-0.147988,6.0,0.03 to 0.24


## 3\. County Health Data: Individual Metrics

Short description: 2022 County Health Release Analytic Data as published by the University of Wisconsin Population Health Institute\. This specific file lists 74 metrics for each county that span across length and quality of life, health infrastructure, the physical environment, and social and economic factors\. They are measured by different units \(e\.g\. some as per capita counts, some as proportions\), hence the standardization by z\-scores when they were factored into the overall rankings above\. Greater detail on their sources can be found in their technical document: https://www\.countyhealthrankings\.org/sites/default/files/media/document/2022%20Analytic%20Documentation\.pdf 

Location: County Health Rankings Data and Documentation can be found here: https://www\.countyhealthrankings\.org/health\-data/methodology\-and\-sources/data\-documentation 
File Link: From that webpage, the metrics csv file can be downloaded via this link: https://www\.countyhealthrankings\.org/sites/default/files/media/document/analytic\_data2022\.csv

Method: Use file link and read in via pandas read\_csv method\.

Key Features:
fipscode: County FIPS Code \(used to merge with other datasets\)
county: County Name
state: State Abbreviation
v\#\#\#\_rawvalue: 1 of 74 metrics related to a county's health and community conditions

### Ingesting the County Health Metrics

In [13]:
def read_in_chd_metrics():
    # Identify the download URL
    base_url = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2022.csv"
    
    # User-Agent for accessing CHD URLs. 
    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36'
    }

    # SELECTING WHICH COLUMNS TO INCLUDE ------------------
    # Regex pattern pulls only the columns with raw values, excluding columns such as proportion numerators, 
    # proportion denominators, confidence intervals, and values pre-segmented by demographic characteristic
    raw_value_regex = r'.+_rawvalue$'
    # Identifying data for the county, the most important of which is the fipscode. 
    id_data_cols = ['statecode', 'countycode', 'fipscode', 'state', 'county']

    # Read in the data
    chd_metrics = pd.read_csv(base_url, header=1, dtype=object, storage_options=headers, 
                         usecols=lambda col : col in id_data_cols or bool(re.match(raw_value_regex, col)))
    
    return chd_metrics

In [14]:
chd_metrics_df = read_in_chd_metrics()

### Cleaning the County Health Metrics

In [15]:
chd_metrics_config = pd.read_csv('data/chd_metrics_config_OLS1.csv') # this file was created manually

def clean_chd_metrics(chd_df=chd_metrics_df, all_cols=True, na_threshold=0.15):

    # Exclude state- and country-level data
    chd_df = chd_df[chd_df['countycode'] != '000'].copy()

    if all_cols == False:        
        # # Exclude metrics in which more than na_threshold of counties do not report data. 
        # # Not recommended due to the missing-not-at-random nature of the data. 
        chd_df = chd_df.dropna(axis=1, thresh=len(chd_df) * (1-na_threshold))

    # Changes all numeric columns to type int or float. 
    numeric_cols = chd_df.columns[chd_df.columns.str.endswith("_rawvalue")]
    chd_df[numeric_cols] = chd_df[numeric_cols].apply(pd.to_numeric, errors=error_handling)

    if all_cols == False:
        # Uses the config table to remove unwanted columns 
        chd_config_keep = chd_metrics_config[chd_metrics_config['include'] == True]
    else:
        chd_config_keep = chd_metrics_config 
    
    # Use the config table to rename columns to more intuitive variable names. 
    rename_map = dict(zip(chd_config_keep['old_v_name'], chd_config_keep['new_v_name']))
    chd_df = chd_df.loc[:,chd_config_keep['old_v_name']].rename(columns=rename_map)

    chd_df['FIPS'] = chd_df['fipscode'].astype(str).str.zfill(5)

    return chd_df

In [16]:
cleaned_chd_metrics_df = clean_chd_metrics()

cleaned_chd_metrics_df.to_csv('data/cleaned_chd_metrics_2022.csv')

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fba226e9-ff4f-4eec-8dfc-4f877d29c8c6' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>